[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap08/cap08.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)


## 💻 **Parte Prática com Exercícios de Programação**

🚧 **Em construção!**

A presente lista de exercícios de programação (EP) consolida as formulações teóricas apresentadas ao longo do Capítulo 8 — Correspondência de Características, Detecção de Objetos e Segmentação Clássica — por meio de uma trilha prática aplicada. Assim como no capítulo anterior, os EPs isolam as **grandezas intermediárias** de cada técnica — a distância entre descritores binários, os termos de uma imagem integral, a contagem de *inliers* de um modelo candidato, a sobreposição entre caixas delimitadoras e o rótulo de cada componente conectado — permitindo validar manualmente cada etapa do raciocínio sem depender do OpenCV nem de imagens externas.

O encadeamento dos exercícios reproduz o fluxo conceitual do capítulo: inicia-se com a **distância de Hamming**, coração da correspondência de descritores binários como o ORB; avança-se para a contagem de ***inliers*** que sustenta o **RANSAC** na estimação robusta de uma homografia; prossegue-se com a **imagem integral**, o truque computacional que torna o Haar Cascade viável em tempo real; aprofunda-se em **IoU e Supressão de Não-Máximos**, o pós-processamento comum a praticamente todo detector de objetos; e conclui-se com a **rotulagem de componentes conectados**, a abordagem clássica — e suas limitações — para segmentar instâncias individuais em uma máscara binária.

> ### ❗ Diretrizes para a Resolução dos Exercícios de Programação
>
> Em todos os exercícios deste capítulo, as etapas de discretização ou arredondamento numérico devem empregar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*), mitigando ambiguidades em valores com fração exatamente igual a $0{,}5$. Salvo indicação explícita em contrário: (i) caixas delimitadoras são especificadas no formato canto-a-canto $(x_{min}, y_{min}, x_{max}, y_{max})$; (ii) matrizes e imagens usam indexação a partir de $0$, com a convenção `[linha][coluna]`; e (iii) comparações de limiar seguem exatamente a convenção descrita em cada exercício — preste atenção especial a se o limiar é excludente ($<$) ou inclusivo ($\le$), pois isso varia entre os exercícios, tal como no código de referência do próprio capítulo.


### 🎯 Objetivo deste Caderno

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.


#### Download

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:


In [17]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")


✅ Ambiente pronto. Morph: 1.1.2 | TestSuite: 1.1.2


#### Executando os Testes
Para rodar os testes, execute `TestSuite("EP08_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como string numa variável `codigo`:

```python
codigo = """
# ... seu código aqui ...
"""
TestSuite("EP08_01").run_code(codigo)
```


### EP08_01 🟢 Distância de Hamming e Correspondência de Descritores Binários

O ORB, usado no Projeto Prático 1 deste capítulo, descreve a vizinhança de cada ponto de interesse como uma sequência de bits — e, por isso, a comparação entre dois descritores não usa a distância euclidiana do k-NN do Capítulo 7, e sim a **distância de Hamming**: o número de posições em que os bits diferem. Antes de chamar `cv2.BFMatcher(cv2.NORM_HAMMING)`, você foi encarregado de implementar manualmente essa correspondência (*matching*) por força bruta — a mesma etapa que, executada internamente pelo OpenCV, precede a estimação robusta da homografia por RANSAC.

#### 📋 Diretrizes de Implementação

1. **Quantidades:** Ler os inteiros $N$ e $M$ — número de descritores extraídos da imagem A e da imagem B, respectivamente.
2. **Descritores de A:** Ler $N$ linhas, cada uma contendo um descritor binário (uma *string* de caracteres `0` e `1`, todos do mesmo comprimento).
3. **Descritores de B:** Ler $M$ linhas, no mesmo formato.
4. **Limiar:** Ler o inteiro $\tau$ — distância de Hamming máxima aceitável para considerar uma correspondência válida.
5. **Distância de Hamming:** Para dois descritores binários $a$ e $b$ de mesmo comprimento,
$$
d_H(a, b) = \sum_{k} \mathbb{1}[a_k \neq b_k],
$$
   ou seja, a contagem de posições em que os bits diferem.
6. **Correspondência por vizinho mais próximo:** Para cada descritor $a_i$ de A ($i$ na ordem de leitura, começando em $0$), calcule sua distância de Hamming a **todos** os descritores de B e encontre o de menor distância. Em caso de empate entre dois ou mais descritores de B com a mesma distância mínima, escolha o de **menor índice**.
7. **Filtragem pelo limiar:** Se a menor distância encontrada for $\le \tau$, a correspondência é válida; caso contrário, $a_i$ não possui correspondência.
8. **Saída:** Para cada $i$ de $0$ a $N-1$, na ordem de leitura, imprimir uma linha: `i j d` se houver correspondência válida (onde $j$ é o índice do descritor de B escolhido e $d$ sua distância), ou `i -1` caso contrário. Ao final, imprimir `Total correspondências válidas: X`.

#### 📌 Restrições Computacionais

* **Mesmo comprimento:** todos os descritores (de A e de B) têm exatamente o mesmo número de bits.
* **Força bruta:** compare cada descritor de A a **todos** os de B — não é necessário nenhum tipo de indexação ou estrutura de aceleração.
* **Desempate por menor índice em B**, e **nunca** por ordem de leitura de A (que já é natural, pois cada $a_i$ é tratado de forma independente).

#### 🧠 Fundamentação Teórica

| Elemento | Papel na correspondência ORB |
|---|---|
| Descritor binário (BRIEF) | Cada bit é o resultado de uma comparação de intensidade entre dois pixels da vizinhança |
| Distância de Hamming | Métrica de dissimetria entre *strings* binárias; muito mais rápida de calcular que a distância euclidiana (operação XOR + contagem de bits) |
| Vizinho mais próximo | Critério de correspondência: cada ponto de A é pareado ao ponto de B com descritor mais similar |
| Limiar $\tau$ | Filtra correspondências pouco confiáveis antes mesmo do RANSAC — mas, como discutido no capítulo, algumas correspondências incorretas ainda passam, exigindo a robustez do RANSAC |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiros $N$ e $M$.
* Próximas $N$ linhas: um descritor binário por linha (*string* de `0`s e `1`s).
* Próximas $M$ linhas: um descritor binário por linha, no mesmo formato.
* Última linha: Inteiro $\tau$.

**Saída:**

* $N$ linhas, uma por descritor de A, no formato `i j d` ou `i -1`.
* Última linha: `Total correspondências válidas: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3 3<br>10101010<br>11110000<br>00001111<br>10101011<br>00001110<br>11111111<br>2 | 0 0 1<br>1 -1<br>2 1 1<br>Total correspondências válidas: 2 | O descritor `11110000` não encontra correspondência: seu vizinho mais próximo está a distância 4, acima do limiar $\tau=2$. |


In [18]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0801" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Distância de Hamming</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 descritores de 8 bits</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Clique em qualquer bit do descritor B para invertê-lo e observe a distância de Hamming mudar em tempo real.</p>
    <div style="display:flex;flex-direction:column;gap:14px;align-items:center;">
      <div>
        <div style="font-size:11px;color:#777;margin-bottom:4px;">Descritor A (fixo)</div>
        <div id="ep0801_a" style="display:grid;grid-template-columns:repeat(8,36px);gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;margin-bottom:4px;">Descritor B (clique para inverter um bit)</div>
        <div id="ep0801_b" style="display:grid;grid-template-columns:repeat(8,36px);gap:4px;"></div>
      </div>
    </div>
    <div id="ep0801_debug" style="margin-top:18px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var A = [1,0,1,0,1,0,1,0];
    var B = [1,0,1,0,1,0,1,1];

    var aEl = root.querySelector('#ep0801_a');
    var bEl = root.querySelector('#ep0801_b');
    var dbg = root.querySelector('#ep0801_debug');

    function estiloBit(v, destacado){
      return 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-weight:bold;font-size:15px;cursor:pointer;user-select:none;' +
        (destacado ? 'background:#fde2e1;border:2px solid #e74c3c;color:#a12;' : 'background:#dbeafe;border:1px solid #93c5fd;color:#1e3a8a;');
    }

    function render(){
      aEl.innerHTML = ''; bEl.innerHTML = '';
      var dist = 0;
      for(var k=0;k<8;k++){
        var diff = A[k] !== B[k];
        if(diff) dist++;
        var da = document.createElement('div');
        da.style.cssText = estiloBit(A[k], diff);
        da.textContent = A[k];
        aEl.appendChild(da);

        var db = document.createElement('div');
        db.style.cssText = estiloBit(B[k], diff);
        db.textContent = B[k];
        db.addEventListener('click', (function(idx){ return function(){ B[idx] = 1-B[idx]; render(); }; })(k));
        bEl.appendChild(db);
      }
      dbg.textContent = 'A = '+A.join('')+'   B = '+B.join('')+'   →   distância de Hamming = '+dist;
    }
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0801');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 8.1:** Simulador: Distância de Hamming entre Dois Descritores Binários


In [19]:
%%writefile EP08_01.py
# Código Python


Overwriting EP08_01.py


In [20]:
TestSuite("EP08_01.py").run()


### EP08_02 🟡 Homografia e RANSAC: A Votação por *Inliers*

O RANSAC, apresentado na seção "Modelagem Matemática: Homografia e RANSAC", repete um ciclo de três passos — sortear uma amostra mínima, estimar um modelo candidato, e contar quantas correspondências são consistentes com ele (os ***inliers***) — mantendo ao final o modelo mais votado. A etapa de estimação do modelo a partir de 4 pontos (passo 2) envolve álgebra linear que foge ao escopo deste EP; aqui, você recebe diretamente um conjunto de homografias **já candidatas** — como se cada uma tivesse sido estimada a partir de uma amostra aleatória diferente — e é encarregado de reproduzir exatamente o passo decisivo do algoritmo: **aplicar cada modelo a todas as correspondências e contar seus *inliers***, escolhendo o vencedor.

#### 📋 Diretrizes de Implementação

1. **Correspondências:** Ler o inteiro $N$ e, em seguida, $N$ linhas com quatro reais cada, $x\ y\ x'\ y'$ — um ponto da imagem A e seu correspondente (possivelmente incorreto) na imagem B, exatamente como produzido pela etapa de *matching* do EP08_01.
2. **Modelos candidatos:** Ler o inteiro $K$ (número de homografias candidatas) e o real $\varepsilon$ (limiar de erro de reprojeção). Em seguida, ler $K$ linhas, cada uma com nove reais $h_{11}\ h_{12}\ h_{13}\ h_{21}\ h_{22}\ h_{23}\ h_{31}\ h_{32}\ h_{33}$ — os elementos da matriz $H$ candidata, em ordem de leitura por linha (*row-major*).
3. **Reprojeção:** Para cada correspondência $(x,y,x',y')$ e cada modelo candidato $H_k$, calcular o ponto projetado
$$
\begin{bmatrix} \hat x \\ \hat y \\ \hat w \end{bmatrix} = H_k \begin{bmatrix} x \\ y \\ 1 \end{bmatrix},
\qquad
(\hat x / \hat w,\ \hat y / \hat w)\ \text{é o ponto projetado.}
$$
4. **Erro de reprojeção:** $e = \sqrt{(\hat x/\hat w - x')^2 + (\hat y /\hat w - y')^2}$.
5. **Contagem de *inliers*:** Uma correspondência é um *inlier* do modelo $H_k$ se $e \le \varepsilon$.
6. **Seleção do melhor modelo:** O modelo vencedor é o de maior número de *inliers*; em caso de empate, escolha o de **menor índice** $k$ (o primeiro encontrado durante o ciclo iterativo do RANSAC).
7. **Saída:** Para cada modelo $k$ de $0$ a $K-1$, na ordem de leitura, imprimir `Modelo k: I inliers`. Ao final, imprimir `Melhor modelo: k_best com I_best inliers`.

#### 📌 Restrições Computacionais

* **Comparação inclusiva:** um erro de reprojeção **exatamente igual** a $\varepsilon$ conta como *inlier* ($e \le \varepsilon$).
* **Sem estimação de $H$:** as matrizes já são fornecidas prontas — não é necessário (nem esperado) resolver nenhum sistema linear.
* **Empate resolvido pelo menor índice**, refletindo o comportamento natural de um algoritmo iterativo que percorre os modelos em ordem e só substitui o melhor encontrado até então quando um novo modelo o **supera estritamente**.

#### 🧠 Fundamentação Teórica

| Elemento | Papel no RANSAC |
|---|---|
| Amostra mínima (4 pares) | Suficiente para determinar os 8 graus de liberdade de uma homografia |
| Modelo candidato $H_k$ | Estimado a partir de uma amostra mínima; pode ser bom ou ruim, dependendo se a amostra continha *outliers* |
| Erro de reprojeção | Mede o quão bem o modelo "prevê" cada correspondência observada |
| *Inlier* vs. *outlier* | Correspondências consistentes com o modelo vencedor (*inliers*) vs. as demais, tipicamente correspondências incorretas do *matching* |
| Refinamento final | Na prática, após escolher o melhor modelo, o RANSAC o recalcula usando **apenas** seus *inliers* — passo não exigido neste EP |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$.
* Próximas $N$ linhas: quatro reais $x\ y\ x'\ y'$.
* Próxima linha: Inteiro $K$ e real $\varepsilon$.
* Próximas $K$ linhas: nove reais (elementos de $H_k$, *row-major*).

**Saída:**

* $K$ linhas no formato `Modelo k: I inliers`.
* Última linha: `Melhor modelo: k_best com I_best inliers`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5<br>0 0 0 0<br>1 1 2 2<br>2 0 4 0<br>0 2 0 4<br>5 5 1 1<br>2 0.5<br>2 0 0 0 2 0 0 0 1<br>1 0 0 0 1 0 0 0 1 | Modelo 0: 4 inliers<br>Modelo 1: 1 inliers<br>Melhor modelo: 0 com 4 inliers | O Modelo 0 (escala ×2) explica corretamente 4 das 5 correspondências; a 5ª, $(5,5)\to(1,1)$, é um *outlier* que nenhum dos dois modelos explica bem. |


In [21]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0802" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: RANSAC — Contagem de Inliers</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 modelo: escala ×2</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">O modelo candidato mapeia (x,y) → (2x,2y). Ajuste o limiar ε e veja quais correspondências (pontos) tornam-se inliers (verde) ou outliers (vermelho).</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">limiar ε</label>
        <span id="ep0802_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">0.50</span>
      </div>
      <input id="ep0802_sl" style="width:100%;accent-color:#2980b9;" max="13" min="0" step="0.25" type="range" value="0.5">
    </div>
    <div id="ep0802_cards" style="display:grid;grid-template-columns:repeat(5,1fr);gap:8px;"></div>
    <div id="ep0802_debug" style="margin-top:18px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var pontos = [
      {x:0,y:0,xp:0,yp:0},
      {x:1,y:1,xp:2,yp:2},
      {x:2,y:0,xp:4,yp:0},
      {x:0,y:2,xp:0,yp:4},
      {x:5,y:5,xp:1,yp:1}
    ];

    var slEl = root.querySelector('#ep0802_sl');
    var vlEl = root.querySelector('#ep0802_vl');
    var cards = root.querySelector('#ep0802_cards');
    var dbg = root.querySelector('#ep0802_debug');

    function render(){
      var eps = parseFloat(slEl.value);
      vlEl.textContent = eps.toFixed(2);
      cards.innerHTML = '';
      var inliers = 0;
      pontos.forEach(function(p, i){
        var px = 2*p.x, py = 2*p.y;
        var erro = Math.sqrt((px-p.xp)*(px-p.xp) + (py-p.yp)*(py-p.yp));
        var dentro = erro <= eps;
        if(dentro) inliers++;
        var div = document.createElement('div');
        div.style.cssText = 'text-align:center;border-radius:10px;padding:10px 6px;font-size:11px;' +
          (dentro ? 'background:#dcfce7;border:1px solid #86efac;color:#14532d;' : 'background:#fee2e2;border:1px solid #fca5a5;color:#991b1b;');
        div.innerHTML = '<div style="font-weight:700;">('+p.x+','+p.y+')→('+p.xp+','+p.yp+')</div>' +
          '<div style="font-family:monospace;margin:4px 0;">erro='+erro.toFixed(2)+'</div>' +
          '<div style="font-weight:700;">'+(dentro?'INLIER':'outlier')+'</div>';
        cards.appendChild(div);
      });
      dbg.textContent = 'ε = '+eps.toFixed(2)+'  |  inliers = '+inliers+' de '+pontos.length;
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0802');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 8.2:** Simulador: RANSAC — Votação por Inliers entre Modelos Candidatos


In [22]:
%%writefile EP08_02.py
# Código Python


Overwriting EP08_02.py


In [23]:
TestSuite("EP08_02.py").run()


### EP08_03 🟢 Imagem Integral: Somas Retangulares em Tempo Constante

O Haar Cascade avalia milhares de características retangulares por janela, em múltiplas posições e escalas — algo inviável em tempo real se cada retângulo exigisse somar seus pixels um a um. A **imagem integral**, definida na seção sobre Haar Cascade, resolve esse problema: uma vez pré-computada, a soma de intensidades de **qualquer** região retangular é obtida com apenas quatro consultas e três operações aritméticas, independentemente do tamanho do retângulo.

Você foi encarregado de implementar essa estrutura do zero: primeiro, calcular a imagem integral a partir da imagem original; em seguida, respondê-la para consultas retangulares arbitrárias.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler as dimensões $H \times W$ da imagem e seus $H \times W$ valores inteiros de intensidade.
2. **Imagem integral:** Calcular, para cada posição $(i,j)$ (indexação a partir de $0$, `[linha][coluna]`),
$$
II(i,j) = \sum_{i' \le i,\ j' \le j} I(i', j'),
$$
   ou seja, a soma de todos os pixels acima e à esquerda de $(i,j)$, incluindo a própria posição.
3. **Consultas:** Ler o inteiro $Q$ e, em seguida, $Q$ linhas, cada uma com quatro inteiros $x_1\ y_1\ x_2\ y_2$ — os cantos superior-esquerdo e inferior-direito de um retângulo, **ambos inclusivos**, com $0 \le x_1 \le x_2 < W$ e $0 \le y_1 \le y_2 < H$.
4. **Soma retangular em O(1):** Para cada consulta, calcular a soma de intensidades dentro do retângulo usando exclusivamente valores já presentes em $II$ (sem percorrer os pixels originais):
$$
S(x_1,y_1,x_2,y_2) = II(y_2,x_2) - II(y_2, x_1{-}1) - II(y_1{-}1, x_2) + II(y_1{-}1, x_1{-}1),
$$
   tratando qualquer termo com índice de linha ou coluna igual a $-1$ como $0$.
5. **Saída:** Primeiro, imprimir a imagem integral completa — $H$ linhas com $W$ inteiros cada. Em seguida, para cada consulta, imprimir um único inteiro: a soma da região correspondente.

#### 📌 Restrições Computacionais

* **Não recalcule por força bruta:** a resposta a cada consulta deve usar a fórmula de quatro termos sobre $II$, não uma soma direta dos pixels do retângulo (ainda que o resultado numérico seja o mesmo, o objetivo do exercício é justamente essa técnica).
* **Retângulos com coordenadas inclusivas:** $(x_1,y_1)$ e $(x_2,y_2)$ pertencem à região somada.
* **Tratamento de borda:** ao consultar $II$ com índice $-1$ (quando $x_1=0$ ou $y_1=0$), utilize o valor $0$.

#### 🧠 Fundamentação Teórica

| Elemento | Papel no Haar Cascade |
|---|---|
| Imagem integral $II$ | Pré-computada uma única vez por imagem, em tempo $O(HW)$ |
| Consulta em O(1) | Cada característica Haar (diferença entre somas de regiões retangulares) é avaliada com poucas operações, independentemente da área do retângulo |
| Escalabilidade | É essa constância que viabiliza avaliar milhares de características, em múltiplas posições e escalas, em tempo real |
| Princípio de inclusão-exclusão | Os quatro termos da fórmula somam a região desejada e subtraem exatamente as áreas contadas em excesso |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiros $H$ e $W$.
* Próximas $H$ linhas: $W$ inteiros cada (imagem original).
* Próxima linha: Inteiro $Q$.
* Próximas $Q$ linhas: quatro inteiros $x_1\ y_1\ x_2\ y_2$.

**Saída:**

* $H$ linhas com $W$ inteiros cada (a imagem integral).
* $Q$ linhas, uma por consulta, com a soma da região correspondente.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>1<br>0 0 2 2 | 1 3 6<br>5 12 21<br>12 27 45<br>45 | A consulta cobre a imagem inteira; a soma coincide com $II(2,2)$ e com a soma de todos os 9 valores. |
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>2<br>1 1 2 2<br>0 0 1 1 | 1 3 6<br>5 12 21<br>12 27 45<br>28<br>12 | A primeira consulta usa os quatro termos da fórmula; a segunda coincide diretamente com $II(1,1)$, pois começa na origem. |


In [24]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0803" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Imagem Integral</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 consulta em 4 termos</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Escolha o retângulo (i,j) inferior-direito da consulta, sempre a partir da origem (0,0) — a região destacada usa apenas o valor de II nesse canto.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">canto inferior-direito da consulta</label>
        <span id="ep0803_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(2,2)</span>
      </div>
      <input id="ep0803_sl" style="width:100%;accent-color:#2980b9;" max="8" min="0" step="1" type="range" value="8">
    </div>
    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Imagem original I (3×3)</div>
        <div id="ep0803_grid" style="display:grid;grid-template-columns:repeat(3,48px);gap:3px;"></div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Imagem integral II (3×3)</div>
        <div id="ep0803_ii" style="display:grid;grid-template-columns:repeat(3,48px);gap:3px;"></div>
      </div>
    </div>
    <div id="ep0803_debug" style="margin-top:18px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var I = [[1,2,3],[4,5,6],[7,8,9]];
    var II = [[0,0,0],[0,0,0],[0,0,0]];
    for(var i=0;i<3;i++) for(var j=0;j<3;j++){
      II[i][j] = I[i][j] +
        (i>0?II[i-1][j]:0) + (j>0?II[i][j-1]:0) - (i>0&&j>0?II[i-1][j-1]:0);
    }

    var slEl = root.querySelector('#ep0803_sl');
    var vlEl = root.querySelector('#ep0803_vl');
    var gridEl = root.querySelector('#ep0803_grid');
    var iiEl = root.querySelector('#ep0803_ii');
    var dbg = root.querySelector('#ep0803_debug');

    function render(){
      var pos = parseInt(slEl.value);
      var i2 = Math.floor(pos/3), j2 = pos%3;
      vlEl.textContent = '('+i2+','+j2+')';
      gridEl.innerHTML = ''; iiEl.innerHTML = '';
      for(var r=0;r<3;r++){
        for(var c=0;c<3;c++){
          var dentro = (r<=i2 && c<=j2);
          var d1 = document.createElement('div');
          d1.style.cssText = 'width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;' +
            (dentro ? 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;' : 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;');
          d1.textContent = I[r][c];
          gridEl.appendChild(d1);

          var destaqueII = (r===i2 && c===j2);
          var d2 = document.createElement('div');
          d2.style.cssText = 'width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;' +
            (destaqueII ? 'background:#dbeafe;border:2px solid #2980b9;font-weight:bold;color:#0d47a1;' : 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;');
          d2.textContent = II[r][c];
          iiEl.appendChild(d2);
        }
      }
      dbg.textContent = 'Consulta de (0,0) a ('+i2+','+j2+')  →  soma = II('+i2+','+j2+') = '+II[i2][j2];
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0803');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 8.3:** Simulador: Imagem Integral e Consulta Retangular em O(1)


In [25]:
%%writefile EP08_03.py
# Código Python


Overwriting EP08_03.py


In [26]:
TestSuite("EP08_03.py").run()


### EP08_04 🟡 IoU e Supressão de Não-Máximos (NMS)

A figura desta seção mostrou o efeito da Supressão de Não-Máximos sobre um conjunto de caixas produzidas por um detector do tipo *sliding window*: múltiplas detecções redundantes por objeto foram reduzidas a uma única caixa por objeto. Você foi encarregado de reimplementar, byte a byte, as duas funções que produziram aquele resultado — `calcular_iou` e `supressao_nao_maximos` — para confirmar, com suas próprias mãos, exatamente os números que o capítulo apresentou.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler o inteiro $N$ (número de caixas) e o real $\tau$ (limiar de IoU). Em seguida, ler $N$ linhas, cada uma com cinco reais $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.
2. **Interseção sobre União:** Para duas caixas $A$ e $B$,
$$
\mathrm{IoU}(A,B) = \frac{\text{área}(A \cap B)}{\text{área}(A) + \text{área}(B) - \text{área}(A \cap B)},
$$
   com área de interseção nula quando as caixas não se sobrepõem.
3. **Algoritmo de NMS** (exatamente como descrito no capítulo):
   a. Ordene as caixas por `score` decrescente (empates mantêm a ordem de leitura original).
   b. Selecione a caixa de maior pontuação entre as restantes; adicione-a à saída e remova-a da lista.
   c. Descarte, da lista restante, **todas** as caixas cujo IoU com a caixa selecionada seja **maior ou igual** a $\tau$ — apenas as caixas com $\mathrm{IoU} < \tau$ permanecem candidatas.
   d. Repita (b)–(c) até que a lista de restantes esteja vazia.
4. **Saída:** Para cada caixa mantida, na ordem em que foi selecionada, imprimir seu índice original (posição de leitura, a partir de $0$) e seu `score`, com 2 casas decimais. Ao final, imprimir `Total mantidas: X`.

#### 📌 Restrições Computacionais

* **Atenção ao sentido do limiar:** ao contrário do que se poderia supor, uma caixa é **suprimida** quando $\mathrm{IoU} \ge \tau$ (não apenas quando $\mathrm{IoU} > \tau$) — siga exatamente esse critério, o mesmo do código de referência do capítulo.
* **Índices originais:** a saída referencia a posição de leitura de cada caixa na entrada, não sua posição após a ordenação por `score`.
* **Área sem soma de 1 pixel:** use área $= (x_{max}-x_{min}) \times (y_{max}-y_{min})$, exatamente como no capítulo (sem o ajuste "+1" às vezes usado em outras convenções).

#### 🧠 Fundamentação Teórica

| Elemento | Papel no pós-processamento |
|---|---|
| IoU | Quantifica a sobreposição espacial entre duas caixas delimitadoras |
| *Sliding window* (Haar Cascade) | Produz tipicamente várias detecções sobrepostas para o mesmo objeto, em posições e escalas próximas |
| Limiar $\tau$ | Controla a agressividade da supressão: baixo demais funde objetos próximos; alto demais deixa passar redundâncias |
| Ordenação por `score` | Garante que, entre caixas redundantes, a de maior confiança sempre sobrevive |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$ e real $\tau$.
* Próximas $N$ linhas: cinco reais $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.

**Saída:**

* Uma linha por caixa mantida, na ordem de seleção: `índice score` (score com 2 casas decimais).
* Última linha: `Total mantidas: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5 0.4<br>50 50 150 150 0.90<br>60 55 155 145 0.75<br>58 60 160 150 0.60<br>300 300 400 420 0.95<br>310 305 395 415 0.70 | 3 0.95<br>0 0.90<br>Total mantidas: 2 | Exatamente o exemplo da figura do capítulo: 5 caixas redundantes (2 objetos) tornam-se 2 detecções finais. O IoU entre a 1ª e a 2ª caixas é $\approx 0{,}775$, bem acima de $\tau=0{,}4$. |


In [27]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0804" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: IoU e Supressão de Não-Máximos</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 supressão se IoU ≥ τ</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">A caixa azul (score maior) já foi selecionada. Ajuste a sobreposição e o limiar τ para ver se a caixa vermelha (candidata) é suprimida.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">deslocamento da candidata</label><span id="ep0804_dx_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">3</span></div>
        <input id="ep0804_dx" style="width:100%;accent-color:#2980b9;" max="10" min="0" step="1" type="range" value="3">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">limiar τ</label><span id="ep0804_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">0.40</span></div>
        <input id="ep0804_tau" style="width:100%;accent-color:#2980b9;" max="0.9" min="0.1" step="0.05" type="range" value="0.4">
      </div>
    </div>
    <div style="position:relative;width:100%;height:160px;background:#fafafa;border:1px solid #ddd;border-radius:12px;margin-bottom:16px;">
      <div id="ep0804_boxA" style="position:absolute;border:2px solid #2980b9;background:rgba(41,128,185,0.25);border-radius:4px;"></div>
      <div id="ep0804_boxB" style="position:absolute;border:2px solid #e74c3c;background:rgba(231,76,60,0.25);border-radius:4px;"></div>
    </div>
    <div id="ep0804_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var dxEl = root.querySelector('#ep0804_dx'), dxvEl = root.querySelector('#ep0804_dx_v');
    var tauEl = root.querySelector('#ep0804_tau'), tauvEl = root.querySelector('#ep0804_tau_v');
    var boxA = root.querySelector('#ep0804_boxA'), boxB = root.querySelector('#ep0804_boxB');
    var dbg = root.querySelector('#ep0804_debug');
    var ESCALA = 10;
    var A = {x1:5, y1:3, x2:15, y2:13};

    function iou(a,b){
      var ix1=Math.max(a.x1,b.x1), iy1=Math.max(a.y1,b.y1);
      var ix2=Math.min(a.x2,b.x2), iy2=Math.min(a.y2,b.y2);
      var iw=Math.max(0, ix2-ix1), ih=Math.max(0, iy2-iy1);
      var inter = iw*ih;
      var areaA=(a.x2-a.x1)*(a.y2-a.y1), areaB=(b.x2-b.x1)*(b.y2-b.y1);
      return inter/(areaA+areaB-inter);
    }

    function render(){
      var dx = parseInt(dxEl.value);
      var tau = parseFloat(tauEl.value);
      dxvEl.textContent = dx; tauvEl.textContent = tau.toFixed(2);
      var B = {x1:5+dx, y1:3+dx*0.4, x2:15+dx, y2:13+dx*0.4};

      boxA.style.left = (A.x1*ESCALA)+'px'; boxA.style.top = (A.y1*ESCALA)+'px';
      boxA.style.width = ((A.x2-A.x1)*ESCALA)+'px'; boxA.style.height = ((A.y2-A.y1)*ESCALA)+'px';
      boxB.style.left = (B.x1*ESCALA)+'px'; boxB.style.top = (B.y1*ESCALA)+'px';
      boxB.style.width = ((B.x2-B.x1)*ESCALA)+'px'; boxB.style.height = ((B.y2-B.y1)*ESCALA)+'px';

      var val = iou(A,B);
      var suprimida = val >= tau;
      dbg.textContent = 'IoU(A,B) = '+val.toFixed(4)+'  |  τ = '+tau.toFixed(2)+'  →  candidata (vermelha) ' + (suprimida ? 'SUPRIMIDA (IoU ≥ τ)' : 'MANTIDA (IoU < τ)');
    }
    dxEl.addEventListener('input', render);
    tauEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0804');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 8.4:** Simulador: IoU e Supressão de Não-Máximos


In [28]:
%%writefile EP08_04.py
# Código Python


Overwriting EP08_04.py


In [29]:
TestSuite("EP08_04.py").run()


### EP08_05 🔴 Rotulagem de Componentes Conectados: Segmentação Clássica de Instâncias

O exemplo de segmentação clássica deste capítulo separou "instâncias" de moedas simplesmente pela sua desconexão espacial na máscara binária resultante da limiarização de Otsu. Essa etapa final — rotular cada componente conectado com um identificador de instância — é exatamente o que você foi encarregado de implementar aqui, do zero, sobre uma máscara binária já pronta (0 = fundo, 1 = objeto), como se fosse uma reimplementação manual de `cv2.connectedComponents`.

Este exercício também expõe, de forma muito concreta, a limitação discutida no capítulo: o resultado depende inteiramente de como se define "vizinhança" entre pixels — e, como você verá no segundo exemplo, dois pixels em diagonal podem ser considerados a mesma instância ou instâncias diferentes, dependendo exclusivamente da **conectividade** escolhida, não de qualquer noção semântica de objeto.

#### 📋 Diretrizes de Implementação

1. **Entrada:** Ler as dimensões $H \times W$ da máscara binária e seus $H \times W$ valores ($0$ ou $1$).
2. **Conectividade:** Ler o inteiro $c \in \{4, 8\}$. Na conectividade $4$, os vizinhos de $(i,j)$ são $(i{-}1,j)$, $(i{+}1,j)$, $(i,j{-}1)$ e $(i,j{+}1)$. Na conectividade $8$, somam-se as quatro diagonais: $(i{-}1,j{-}1)$, $(i{-}1,j{+}1)$, $(i{+}1,j{-}1)$ e $(i{+}1,j{+}1)$.
3. **Descoberta de componentes:** Percorrendo a máscara em varredura linha a linha, da esquerda para a direita e de cima para baixo, sempre que um pixel de valor $1$ ainda sem rótulo for encontrado, ele inicia um **novo componente**: atribua a ele o próximo rótulo disponível (o primeiro componente descoberto recebe o rótulo $1$, o segundo o rótulo $2$, e assim por diante) e propague esse mesmo rótulo a todos os pixels de valor $1$ alcançáveis a partir dele por uma cadeia de vizinhos (de acordo com a conectividade escolhida) — por busca em largura, profundidade, ou *union-find*, à sua escolha.
4. **Pixels de fundo:** permanecem com rótulo $0$ e não pertencem a nenhuma instância.
5. **Saída:** Primeiro, imprimir o mapa de rótulos completo — $H$ linhas com $W$ inteiros cada. Em seguida, para cada rótulo $\ell$ de $1$ a $K$ (na ordem de descoberta), imprimir `Instância l: A pixels`, onde $A$ é a quantidade de pixels com aquele rótulo. Por fim, imprimir `Total de instâncias: K`.

#### 📌 Restrições Computacionais

* **Ordem de descoberta = ordem de varredura:** os rótulos são numerados na ordem em que cada novo componente é encontrado pela varredura linha a linha, não por tamanho ou posição.
* **Conectividade explícita:** dois pixels de valor $1$ só pertencem à mesma instância se existir uma cadeia de vizinhos **de acordo com $c$** ligando um ao outro — não use a conectividade oposta por engano.
* **Máscara binária pura:** todos os valores de entrada são exatamente $0$ ou $1$.

#### 🧠 Fundamentação Teórica

| Elemento | Papel na segmentação clássica de instâncias |
|---|---|
| Limiarização (Otsu, Cap. 4) | Etapa anterior que produz a máscara binária a partir da imagem de intensidade |
| Componente conectado | Cada instância é definida **apenas** por conectividade espacial dos pixels de objeto, sem qualquer noção de forma, classe ou aparência |
| Conectividade 4 vs. 8 | Parâmetro que altera o resultado: sob conectividade 8, dois blobs unidos apenas na diagonal tornam-se uma única instância |
| Limitação central | A técnica funde instâncias que se tocam ou se sobrepõem (mesmo que sejam objetos claramente distintos), pois não há noção de "objeto" — apenas de "região conectada" |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiros $H$ e $W$.
* Próximas $H$ linhas: $W$ inteiros ($0$ ou $1$) cada.
* Última linha: Inteiro $c$ ($4$ ou $8$).

**Saída:**

* $H$ linhas com $W$ inteiros cada (o mapa de rótulos).
* Uma linha por instância, na ordem de descoberta: `Instância l: A pixels`.
* Última linha: `Total de instâncias: K`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | 0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 2 2<br>0 0 0 0 2 2<br>Instância 1: 4 pixels<br>Instância 2: 4 pixels<br>Total de instâncias: 2 | Dois blocos $2\times2$ claramente separados: o resultado é o mesmo sob conectividade 4 ou 8. |
| 2 2<br>1 0<br>0 1<br>8 | 1 0<br>0 1<br>Instância 1: 2 pixels<br>Total de instâncias: 1 | Sob conectividade 8, os dois pixels em diagonal pertencem à **mesma** instância. Repita este exemplo com $c=4$: o resultado passa a ser 2 instâncias de 1 pixel cada — puramente pela mudança de conectividade, sem qualquer diferença na máscara. |


In [30]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0805" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Componentes Conectados</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 mesma máscara, resultado diferente</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">A mesma máscara (dois pixels em diagonal) — alterne a conectividade e observe o número de instâncias e as cores dos rótulos mudarem.</p>
    <div style="display:flex;gap:8px;justify-content:center;margin-bottom:18px;">
      <button id="ep0805_c4" style="cursor:pointer;padding:6px 16px;border-radius:20px;border:1px solid #ddd;background:#f3f4f6;color:#555;font-weight:bold;font-size:11px;">conectividade 4</button>
      <button id="ep0805_c8" style="cursor:pointer;padding:6px 16px;border-radius:20px;border:1px solid #f0ad4e;background:#fff3cd;color:#7a5c00;font-weight:bold;font-size:11px;">conectividade 8</button>
    </div>
    <div style="display:flex;justify-content:center;">
      <div id="ep0805_grid" style="display:grid;grid-template-columns:repeat(2,56px);gap:4px;"></div>
    </div>
    <div id="ep0805_debug" style="margin-top:18px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var mask = [[1,0],[0,1]];
    var conect = 8;
    var CORES = ['#dbeafe', '#fde68a'];
    var BORDAS = ['#2980b9', '#d97706'];

    var btn4 = root.querySelector('#ep0805_c4');
    var btn8 = root.querySelector('#ep0805_c8');
    var gridEl = root.querySelector('#ep0805_grid');
    var dbg = root.querySelector('#ep0805_debug');

    function rotula(){
      var H = mask.length, W = mask[0].length;
      var labels = [[0,0],[0,0]];
      var atual = 0;
      var viz4 = [[-1,0],[1,0],[0,-1],[0,1]];
      var viz8 = viz4.concat([[-1,-1],[-1,1],[1,-1],[1,1]]);
      var viz = conect === 8 ? viz8 : viz4;
      for(var i=0;i<H;i++){
        for(var j=0;j<W;j++){
          if(mask[i][j]===1 && labels[i][j]===0){
            atual++;
            var fila = [[i,j]];
            labels[i][j] = atual;
            while(fila.length){
              var pos = fila.pop(); var r=pos[0], c=pos[1];
              for(var k=0;k<viz.length;k++){
                var nr=r+viz[k][0], nc=c+viz[k][1];
                if(nr>=0 && nr<H && nc>=0 && nc<W && mask[nr][nc]===1 && labels[nr][nc]===0){
                  labels[nr][nc] = atual;
                  fila.push([nr,nc]);
                }
              }
            }
          }
        }
      }
      return {labels: labels, k: atual};
    }

    function estiloBotoes(){
      btn4.style.background = conect===4 ? '#fff3cd' : '#f3f4f6';
      btn4.style.borderColor = conect===4 ? '#f0ad4e' : '#ddd';
      btn4.style.color = conect===4 ? '#7a5c00' : '#555';
      btn8.style.background = conect===8 ? '#fff3cd' : '#f3f4f6';
      btn8.style.borderColor = conect===8 ? '#f0ad4e' : '#ddd';
      btn8.style.color = conect===8 ? '#7a5c00' : '#555';
    }

    function render(){
      var res = rotula();
      gridEl.innerHTML = '';
      for(var i=0;i<2;i++){
        for(var j=0;j<2;j++){
          var d = document.createElement('div');
          var lab = res.labels[i][j];
          var estilo = 'width:56px;height:56px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-weight:bold;font-size:14px;';
          if(lab === 0){
            estilo += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#aaa;';
          } else {
            estilo += 'background:'+CORES[(lab-1)%2]+';border:2px solid '+BORDAS[(lab-1)%2]+';color:#333;';
          }
          d.style.cssText = estilo;
          d.textContent = mask[i][j] + (lab? ' (r'+lab+')' : '');
          gridEl.appendChild(d);
        }
      }
      estiloBotoes();
      dbg.textContent = 'conectividade = '+conect+'  →  '+res.k+' instância(s) encontrada(s)';
    }
    btn4.addEventListener('click', function(){ conect=4; render(); });
    btn8.addEventListener('click', function(){ conect=8; render(); });
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0805');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 8.5:** Simulador: Rotulagem de Componentes Conectados — Conectividade 4 vs. 8


In [31]:
%%writefile EP08_05.py
# Código Python


Overwriting EP08_05.py


In [32]:
TestSuite("EP08_05.py").run()
